In [1]:
!apt-get update -y
!apt-get install tesseract-ocr -y
!pip install -q transformers datasets accelerate evaluate rouge-score pymupdf pdfplumber Pillow pytesseract
!pip install -q langchain langchain-community langchain-text-splitters faiss-cpu langchain-mistralai sentence-transformers
!pip install -q fastapi uvicorn pyngrok nest-asyncio python-multipart

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,646 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:8 https://cli.github.com/packages stable/main amd64 Packages [356 B]       
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB

In [2]:
import os
import torch
import fitz 
from PIL import Image
import io
import evaluate
import pytesseract
import numpy as np
from tqdm.auto import tqdm
from datasets import load_dataset, DatasetDict
from transformers import (
    NougatProcessor, 
    VisionEncoderDecoderModel, 
    Seq2SeqTrainer, 
    Seq2SeqTrainingArguments,
    default_data_collator
)

device = "cuda" if torch.cuda.is_available() else "cpu"
dataset = load_dataset("marcodsn/arxiv-markdown", split="train")
def check_local_pdf(example):
    pdf_path = os.path.join("/kaggle/input/datasets/danghiennguyen/arxiv-pdf-small/arxiv_dataset_pdfs/", f"{example['arxiv_id']}.pdf")
    return os.path.exists(pdf_path)

dataset = dataset.filter(check_local_pdf)
if len(dataset) == 0:
    raise FileNotFoundError("404 Not Found.")

test_size = 0.15 
valid_size = 0.15
train_test_split = dataset.train_test_split(test_size=test_size, seed=42)
test_dataset = train_test_split['test']
train_valid_split = train_test_split['train'].train_test_split(test_size=valid_size, seed=42)
train_dataset = train_valid_split['train']
valid_dataset = train_valid_split['test']

final_datasets = DatasetDict({
    'train': train_dataset,
    'valid': valid_dataset,
    'test': test_dataset
})

print(f"Train: {len(final_datasets['train'])} mẫu")
print(f"Valid: {len(final_datasets['valid'])} mẫu")
print(f"Test: {len(final_datasets['test'])} mẫu")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/96.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3269 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3269 [00:00<?, ? examples/s]

Train: 69 mẫu
Valid: 13 mẫu
Test: 15 mẫu


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
processor = NougatProcessor.from_pretrained("facebook/nougat-small", use_fast=False)
content_col = 'markdown' if 'markdown' in final_datasets['train'].column_names else 'content'
available_columns = final_datasets['train'].column_names

def preprocess_function(examples):
    pixel_values, labels = [], []
    batch_arxiv_ids = examples['arxiv_id']
    batch_markdowns = examples[content_col]
    
    for arxiv_id, markdown_text in zip(batch_arxiv_ids, batch_markdowns):
        pdf_path = os.path.join("/kaggle/input/datasets/danghiennguyen/arxiv-pdf-small/arxiv_dataset_pdfs/", f"{arxiv_id}.pdf")
        if not os.path.exists(pdf_path): continue
            
        try:
            doc = fitz.open(pdf_path)
            pix = doc.load_page(0).get_pixmap(dpi=150)
            img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
            doc.close()
            
            processed_img = processor(img, return_tensors="pt").pixel_values.squeeze()
            pixel_values.append(processed_img)
            
            tokenized_text = processor.tokenizer(
                str(markdown_text)[:1024], padding="max_length", 
                max_length=512, truncation=True, return_tensors="pt"
            ).input_ids.squeeze()
            tokenized_text[tokenized_text == processor.tokenizer.pad_token_id] = -100
            labels.append(tokenized_text)
        except Exception as e:
            continue
    return {"pixel_values": pixel_values, "labels": labels}

train_dataset = final_datasets['train'].map(preprocess_function, batched=True, remove_columns=available_columns)
valid_dataset = final_datasets['valid'].map(preprocess_function, batched=True, remove_columns=available_columns)
rouge = evaluate.load("rouge")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    labels = np.where(labels != -100, labels, processor.tokenizer.pad_token_id)
    decoded_preds = processor.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = processor.batch_decode(labels, skip_special_tokens=True)
    return rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
model = VisionEncoderDecoderModel.from_pretrained("facebook/nougat-small")
start_token = processor.tokenizer.bos_token_id or processor.tokenizer.convert_tokens_to_ids("<s>")
model.config.decoder_start_token_id = start_token
model.config.decoder.decoder_start_token_id = start_token
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.bos_token_id = processor.tokenizer.bos_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id

model.config.decoder.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder.bos_token_id = processor.tokenizer.bos_token_id
model.config.decoder.eos_token_id = processor.tokenizer.eos_token_id
model.config.use_cache = False
model.config.decoder.use_cache = False 
training_args = Seq2SeqTrainingArguments(
    output_dir="./ocr_finetuned_checkpoints",
    per_device_train_batch_size=2,
    eval_strategy="steps", 
    eval_steps=100, 
    save_steps=100,
    learning_rate=3e-5, 
    num_train_epochs=2, 
    predict_with_generate=True, 
    report_to="none",
    ddp_find_unused_parameters=False
)

trainer = Seq2SeqTrainer(
    model=model, args=training_args, train_dataset=train_dataset, eval_dataset=valid_dataset,
    processing_class=processor.tokenizer, data_collator=default_data_collator, compute_metrics=compute_metrics,
)

trainer.train()
model.config.use_cache = True
model.config.decoder.use_cache = True
trainer.save_model("./best_ocr_model")
processor.save_pretrained("./best_ocr_model")

preprocessor_config.json:   0%|          | 0.00/479 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

Step,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['./best_ocr_model/processor_config.json']

In [4]:
import re

def my_finetuned_ocr_tool_full(pdf_path):
    my_model = VisionEncoderDecoderModel.from_pretrained("./best_ocr_model").to(device)
    my_processor = NougatProcessor.from_pretrained("./best_ocr_model", use_fast=False)
    full_markdown_result = []
    
    with fitz.open(pdf_path) as doc:
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            pix = page.get_pixmap(dpi=150)
            img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
            
            pixel_values = my_processor(img, return_tensors="pt").pixel_values.to(device)
            outputs = my_model.generate(pixel_values, max_new_tokens=1024, bad_words_ids=[[my_processor.tokenizer.unk_token_id]])
            
            sequence = my_processor.batch_decode(outputs, skip_special_tokens=True)[0]
            clean_sequence = sequence.replace("[wgn]", "").replace("[MISSING_PAGE_EMPTY]", "").strip()
            clean_sequence = re.sub(r'\n\s*\n', '\n\n', clean_sequence)
            full_markdown_result.append(f"\n{clean_sequence}")
            
    return "\n\n---\n\n".join(full_markdown_result)

In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_mistralai import ChatMistralAI 
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
llm = ChatMistralAI(
    mistral_api_key="fa6IjG5h17eRqpJ2XEl5xvI4ch2k46Hm", 
    model="mistral-small-latest" 
)

vector_store = None

/tmp/ipykernel_57/3376697371.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_57/3376697371.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import nest_asyncio
import uvicorn
import shutil
import os
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import PlainTextResponse
from pydantic import BaseModel
from pyngrok import ngrok
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class ChatQuery(BaseModel):
    text: str
@app.post("/upload-pdf")
async def upload_pdf(file: UploadFile = File(...)):
    temp_pdf_path = f"/kaggle/working/{file.filename}"
    try:
        with open(temp_pdf_path, "wb") as buffer:
            shutil.copyfileobj(file.file, buffer)
        markdown_result = my_finetuned_ocr_tool_full(temp_pdf_path)
        
        if os.path.exists(temp_pdf_path):
            os.remove(temp_pdf_path)
            
        return PlainTextResponse(content=markdown_result, media_type="text/markdown")
    except Exception as e:
        return PlainTextResponse(content=f"Lỗi xử lý OCR: {str(e)}", status_code=500)
@app.post("/upload-markdown")
async def upload_markdown(file: UploadFile = File(...)):
    global vector_store
    try:
        markdown_content = await file.read()
        markdown_text = markdown_content.decode("utf-8")
        
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        docs = text_splitter.create_documents(
            texts=[markdown_text], 
            metadatas=[{"source": file.filename}]
        )
        
        vector_store = FAISS.from_documents(docs, embeddings)
        return {"status": "success", "message": f"Đã nạp thành công tài liệu: {file.filename} vào AI Agent!"}
    except Exception as e:
        return {"status": "error", "message": f"Lỗi nạp tài liệu: {str(e)}"}
@app.post("/chat")
async def chat_with_document(query: ChatQuery):
    global vector_store
    if vector_store is None:
        return {"answer": "Hệ thống chưa nhận được dữ liệu. Vui lòng chạy OCR và tải file MD lên trước!"}
    
    try:
        retriever = vector_store.as_retriever(search_kwargs={"k": 4})
        relevant_docs = retriever.invoke(query.text)
        context = "\n\n".join([doc.page_content for doc in relevant_docs])
        
        prompt = ChatPromptTemplate.from_template("""
        Bạn là một AI chuyên phân tích, tổng hợp thông tin và đưa ra tóm tắt những thong tin cần thiết. Trả lời dựa TRỰC TIẾP vào CONTEXT dưới đây.
        NỘI DUNG TÀI LIỆU (CONTEXT):
        {context}
        CÂU HỎI NGƯỜI DÙNG:
        {question}
        TRẢ LỜI:
        """)
        
        chain = prompt | llm | StrOutputParser()
        answer = chain.invoke({"context": context, "question": query.text})
        
        return {"answer": answer}
    except Exception as e:
        return {"answer": f"Có lỗi khi AI phân tích: {str(e)}"}

NGROK_AUTH_TOKEN = "3BVlsDwoTkPQgC0RgVKc5d6js8C_5MZCmPKqofZaEotxqtUJ3"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
auth_proxy_tunnel = ngrok.connect(8000)

print(f"LINK API: {auth_proxy_tunnel.public_url}")


nest_asyncio.apply()
config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [57]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


LINK API: https://monsoonal-unbolstered-elizabet.ngrok-free.dev
